In [3]:
from transformers import BertTokenizer, EncoderDecoderModel

import torch

In [4]:
MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    MODEL_NAME,
    MODEL_NAME
)

model.config.decoder.is_decoder = True
model.config.decoder.add_cross_attention = True

model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.bos_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id

model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.bos_token_id = tokenizer.cls_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

def make_input(text, topic_words):
    topic_text = " ".join(topic_words)
    return f"topic: {topic_text} text: {text}"

dataset = [
    {
        "text": "patient diagnosed diabetes received medicine hospital",
        "topic_words": ["patient", "doctor", "medicine", "hospital", "diabetes"],
        "summary": "patient received diabetes treatment"
    },
    {
        "text": "bank approved loan for customer",
        "topic_words": ["bank", "money", "loan", "finance", "customer"],
        "summary": "bank approved loan"
    }
]

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(5):
    total_loss = 0

    for item in dataset:
        src_text = make_input(item["text"], item["topic_words"])
        tgt_text = item["summary"]

        inputs = tokenizer(
            src_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        labels = tokenizer(
            tgt_text,
            return_tensors="pt",
            padding=True,
            truncation=True
        )["input_ids"]

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss={total_loss:.4f}")

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bias', 'bert.e

Epoch 1, Loss=27.4626
Epoch 2, Loss=21.5607
Epoch 3, Loss=15.7244
Epoch 4, Loss=12.2750
Epoch 5, Loss=10.0714


In [5]:
text = "sick patient received medicine doctor"
topic_words = ["patient", "doctor", "medicine", "health", "treatment"]

src_text = make_input(text, topic_words)

inputs = tokenizer(
    src_text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

generated_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=20
)

summary = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print(summary)

patient patient patient patient patient patient patient patient patient patient patient patient patient patient patient patient patient patient patient


** seems like BERT is not for text generation; it is only alone not giving good results.